In [1]:
# Importando bibliotecas
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split #para dividir os dados em treino e teste
from sklearn.compose import ColumnTransformer # para aplicar diferentes transformações em diferentes colunas do dataset
from sklearn.pipeline import Pipeline # para criar um pipeline de pré-processamento e modelagem
from sklearn.impute import SimpleImputer # para lidar com valores ausentes
from sklearn.preprocessing import OneHotEncoder # para codificar variáveis categóricas
from sklearn.preprocessing import StandardScaler # para padronizar os dados

#recarga da base
df = pd.read_csv(r'C:\Users\marciohenrique\Documents\Projetos\github\ml_stat\bases_dados\base_imoveis.csv')

In [3]:
"""Tratamento de valores nulos, para as colunas numéricas, 
vamos substituir os valores nulos pela mediana da coluna,
 e para as colunas categóricas, vamos substituir os valores nulos pela moda da coluna tudo via pipeline.
 exclusão de nulos da variavel alvo (preco_venda) """

colunas = [
    "area_m2",
    "quartos",
    "banheiros",
    "vagas_garagem",
    "idade_anos",
    "localidade",
    "tipo"
]

val_numer = [
    "area_m2",
    "quartos",
    "banheiros",
    "vagas_garagem",
    "idade_anos"
]

val_categ = [
    "localidade",
    "tipo"
]

df = df.dropna(subset=["preco_venda"]).copy() #copia da base de dados para não alterar a original


transf_val_num= Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")), # substitui valores nulos pela mediana
    ]
)


transf_val_cat= Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")), # substitui valores nulos pela moda
    ("onehot", OneHotEncoder(handle_unknown="ignore")) # codifica variáveis categóricas
    ]
)

In [4]:
"""Definindo as variaveis preditoras e a variável alvo
Configuração do percuntual de treino e teste, 80% para treino e 20% para teste"""
X = df[colunas]
y = df["preco_venda"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # 80% treino e 20% teste

"""Preprocessamento dos dados, aplicando as transformações definidas anteriormente nas colunas numéricas e categóricas """
preprocessamento = ColumnTransformer(
    transformers=[
        ("num", transf_val_num, val_numer),
        ("cat", transf_val_cat, val_categ)
    ]
)

**Treino x Teste nos modelos**

Aqui vamos analisar se os modelos estão generalizando bem os dados ou se "decoraram os resultados"
sobretudo nos algoritimos de árvore

In [9]:
# Regressão Linear
from sklearn.linear_model import LinearRegression

modelo_linear = Pipeline(steps=[                    #definindo o pipeline de regressão linear
    ("preprocessamento", preprocessamento),
    ("modelo", LinearRegression())
    ]
)



modelo_linear.fit(X_train, y_train) #treinando o modelo de regressão linear
y_pred_linear = modelo_linear.predict(X_test) #fazendo previsões com o modelo de regressão linear

# Arvore de Decisão
#Usando a mesma base, aplição dos algoritmos de árvore, Decision Tree e Random Forest, para comparar os resultados com o modelo de regressão linear.


from sklearn.tree import DecisionTreeRegressor

max_depth = 10

modelo_arvore = Pipeline(
    steps=[
        ("preprocessamento", preprocessamento),
        (
            "modelo",
            DecisionTreeRegressor(
                max_depth=max_depth,
                random_state=42
            )
        )
    ]
)

modelo_arvore.fit(X_train, y_train)

y_pred_arvore = modelo_arvore.predict(X_test)

# Random Forest

from sklearn.ensemble import RandomForestRegressor
modelo_rf = Pipeline(
    steps=[
        ("processamento", preprocessamento),
        (
            "modelo",
            RandomForestRegressor(
                n_estimators=200,
                max_depth=None,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)
modelo_rf.fit(X_train, y_train)
y_pred_rf = modelo_rf.predict(X_test)



In [11]:
y_pred_linear_train = modelo_linear.predict(X_train)
y_pred_arvore_train = modelo_arvore.predict(X_train)
y_pred_rf_train = modelo_rf.predict(X_train)

Função para não repetir código para verificar os indicadores

In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def avaliar_modelo(y_real, y_pred):
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)

    return mae, rmse, r2

**Analisando os Indicadores(MAE,RMSE e R²) quando os dados são conhecidos pelos modelos**

In [13]:
# ============================
# MÉTRICAS DE TREINO
# ============================

mae_linear_train, rmse_linear_train, r2_linear_train = avaliar_modelo(
    y_train,
    y_pred_linear_train
)

mae_arvore_train, rmse_arvore_train, r2_arvore_train = avaliar_modelo(
    y_train,
    y_pred_arvore_train
)

mae_rf_train, rmse_rf_train, r2_rf_train = avaliar_modelo(
    y_train,
    y_pred_rf_train
)

**Analisando os Indicadores(MAE,RMSE e R²) quando os dados são desconhecidos pelos modelos**

In [14]:
# ============================
# MÉTRICAS DE TESTE
# ============================

mae_linear, rmse_linear, r2_linear = avaliar_modelo(
    y_test,
    y_pred_linear
)

mae_arvore, rmse_arvore, r2_arvore = avaliar_modelo(
    y_test,
    y_pred_arvore
)

mae_rf, rmse_rf, r2_rf = avaliar_modelo(
    y_test,
    y_pred_rf
)

In [15]:
comparacao_treino_teste = pd.DataFrame({
    "Modelo": [
        "Regressão Linear",
        "Árvore de Decisão",
        "Random Forest"
    ],

    "MAE Treino": [
        mae_linear_train,
        mae_arvore_train,
        mae_rf_train
    ],

    "MAE Teste": [
        mae_linear,
        mae_arvore,
        mae_rf
    ],

    "RMSE Treino": [
        rmse_linear_train,
        rmse_arvore_train,
        rmse_rf_train
    ],

    "RMSE Teste": [
        rmse_linear,
        rmse_arvore,
        rmse_rf
    ],

    "R² Treino": [
        r2_linear_train,
        r2_arvore_train,
        r2_rf_train
    ],

    "R² Teste": [
        r2_linear,
        r2_arvore,
        r2_rf
    ]
})

display(comparacao_treino_teste)

,Modelo,MAE Treino,MAE Teste,RMSE Treino,RMSE Teste,R² Treino,R² Teste
0,Regressão Linear,182280.656947,202419.530172,299579.373226,329888.564323,0.837875,0.828785
1,Árvore de Decisão,208702.317118,259312.130173,299086.829165,383307.726518,0.838407,0.768846
2,Random Forest,50698.284363,153691.229536,89261.381553,252716.681364,0.985607,0.899521


**Interpretando resultados**

Nosso fluxo até aqui:



                  TREINO                  TESTE
                    ↓                       ↓
             dados conhecidos       dados não vistos
                    ↓                       ↓
              modelo prevê           modelo prevê
                    ↓                       ↓
              calcula métricas      calcula métricas
                    ↓                       ↓
                    └──────────┬────────────┘
                               ↓
                       COMPARAR RESULTADOS
                               ↓
                   Generalização do modelo

- Implementamos cada modelo de aprendizado supervisionado\
 para previsão de variáveis numéricas Regressão Linear, Árvore de Decisão e Árvores Aleatórias\
 que são respectivamente os algoritmos (Linear Regression, Decision Tree e Random Forest)

- Analisamos os indicadores dos modelos\
MAE, RMSE e R²
- Dividimos a análise desses indicadores entre dados de treino(conhecidos) e de teste(desconhecidos)
Com isso, podemos ennter as capacidades de generalização.\
e verificar se os resultados estão superestimados, subestimados

	Modelo	MAE Treino	MAE Teste	RMSE Treino	RMSE Teste	R² Treino	R² Teste
0	Regressão Linear	182280.656947	202419.530172	299579.373226	329888.564323	0.837875	0.828785
1	Árvore de Decisão	208702.317118	259312.130173	299086.829165	383307.726518	0.838407	0.768846
2	Random Forest	50698.284363	153691.229536	89261.381553	252716.681364	0.985607	0.899521


**Retomando os resultados**
| Modelo | MAE Treino | MAE Teste | RMSE Treino | RMSE Teste | R² Treino | R² Teste |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Regressão Linear** | 182.280,66 | 202.419,53 | 299.579,37 | 329.888,56 | 0,8379 | 0,8288 |
| **Árvore de Decisão** | 208.702,32 | 259.312,13 | 299.086,83 | 383.307,73 | 0,8384 | 0,7688 |
| **Random Forest** | 50.698,28 | 153.691,23 | 89.261,38 | 252.716,68 | 0,9856 | 0,8995 |


- Colunas treino = Dados conhecidos pelos modelos
- Colunas teste = Dados desconhecidos pelos modelos (as predições ou dados novos)
- MAE e RMSE  = Erros cometidos pelos modelos
- R² = Percentual da variancia entre as variáveis(dependentes e independentes)
 explica como o modelo entende os resultados a partir das variaveis independentes
  ou seja Explica quanto preco_imoveis pode ser explicado pelas variações de area, idade, quartos,etc

**OBS** R² não é % de precisão. 

**Regressão Linear** ---> Tanto as métricas de treino quanto as de teste(valor conhecido X valor desconhecido) são bem próximas entre si
Há um equilibrio entre treino e teste, representa boa generalização mas a capacidade de predizer é inferior ao Random Forest

**Árvore de Decisão** ---> Os erros aumentam bastante em dados desconhecidos(teste) R² fica bem reduizdo
Há um problema no modelo para generalizar, tem a menor capacidade preditiva entre os 3

**Árvore de Decisão** ---> Há grandes varições de erro entre teste e treino, mas o menor erro entre os 3 modelos
Há sinais de overfiting, ou seja fica muito especializado e decora mais ao invés de aprender um padrão geral.
Ou seja, o modelo é o melhor candidato porque os 3 indicadores tanto para dados conhecidos quanto para dados desconhecidos são superiores quando comparados aos demais nas mesmas condições.


Até agora vimos esse fluxo:
**treino → teste → métricas → generalização → overfitting**